In [1]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

In [2]:
# ==============================================================================
# 1. LOAD THE 10:1 STRATIFIED RZLT SAMPLE
# ==============================================================================
input_path = os.path.join("..", "Datasets", "RZLT_Samples", "RZLT_AllCodes_Stratified_10to1.geojson")
if not os.path.exists(input_path):
    input_path = "RZLT_AllCodes_Stratified_10to1.geojson"

gdf = gpd.read_file(input_path)

# Ensure Irish Transverse Mercator (EPSG:2157) for meter-accurate Euclidean distances
if gdf.crs != "EPSG:2157":
    gdf = gdf.to_crs(epsg=2157)

if 'centroid_x' not in gdf.columns or 'centroid_y' not in gdf.columns:
    gdf['centroid_x'] = gdf.geometry.centroid.x
    gdf['centroid_y'] = gdf.geometry.centroid.y

parcel_coords = np.column_stack([gdf['centroid_x'], gdf['centroid_y']])
print(f"Loaded {len(gdf):,} RZLT parcels.")

Loaded 6,303 RZLT parcels.


In [5]:
# ==============================================================================
# 2. LOAD NTA GTFS STOPS.TXT & CLASSIFY TRANSIT MODES
# ==============================================================================
gtfs_stops_path = os.path.join("..", "Datasets", "GTFS_All", "stops.txt")  # Update path if needed
if not os.path.exists(gtfs_stops_path):
    raise FileNotFoundError(
        f"Could not find '{gtfs_stops_path}'. Download the TFI NTA GTFS static zip and extract stops.txt."
    )

stops_df = pd.read_csv(gtfs_stops_path, low_memory=False)

# Drop any rows missing valid WGS84 coordinates
stops_df = stops_df.dropna(subset=['stop_lat', 'stop_lon']).copy()

# Convert WGS84 (EPSG:4326) lat/lon -> Irish Transverse Mercator (EPSG:2157)
stops_gdf = gpd.GeoDataFrame(
    stops_df,
    geometry=gpd.points_from_xy(stops_df['stop_lon'], stops_df['stop_lat']),
    crs="EPSG:4326"
).to_crs(epsg=2157)

stops_gdf['x'] = stops_gdf.geometry.x
stops_gdf['y'] = stops_gdf.geometry.y

# Identify Rail / DART / Luas stations vs. Bus stops
# NTA GTFS uses '8220IR...' / '8300IR...' / '8220GA...' for Irish Rail/Luas or explicit keywords in stop_name
stop_id_str = stops_gdf['stop_id'].astype(str).str.upper()
stop_name_str = stops_gdf['stop_name'].astype(str).str.lower()

is_rail_or_luas = (
    stop_id_str.str.contains('IR|LUAS|DART|RAIL', regex=True) |
    stop_name_str.str.contains(r'\b(train station|rail station|luas|dart)\b', regex=True)
)

# Optional: Check GTFS vehicle_type / location_type if present in your feed
if 'vehicle_type' in stops_gdf.columns:
    # GTFS Extended route types: 0/900 = Tram/Luas, 2/100 = Rail
    is_rail_or_luas = is_rail_or_luas | stops_gdf['vehicle_type'].isin([0, 2, 100, 900])

rail_luas_gdf = stops_gdf[is_rail_or_luas].copy()
bus_gdf = stops_gdf[~is_rail_or_luas].copy()

print(f"Total NTA Stops Loaded : {len(stops_gdf):,}")
print(f"  -> Rail / DART / Luas: {len(rail_luas_gdf):,}")
print(f"  -> Bus Stops         : {len(bus_gdf):,}")

Total NTA Stops Loaded : 14,201
  -> Rail / DART / Luas: 191
  -> Bus Stops         : 14,010


C:\Users\musab\AppData\Local\Temp\ipykernel_21304\3014775385.py:32: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  stop_name_str.str.contains(r'\b(train station|rail station|luas|dart)\b', regex=True)


In [6]:
# ==============================================================================
# 3. COMPUTE MULTI-MODAL cKDTree DISTANCES & ACCESSIBILITY FEATURES
# ==============================================================================
# Build separate k-d trees for Bus and Rail/Luas
bus_tree = cKDTree(np.column_stack([bus_gdf['x'], bus_gdf['y']]))
dist_bus, _ = bus_tree.query(parcel_coords, k=1)
gdf['DIST_TO_BUS_M'] = dist_bus

if len(rail_luas_gdf) > 0:
    rail_tree = cKDTree(np.column_stack([rail_luas_gdf['x'], rail_luas_gdf['y']]))
    dist_rail, _ = rail_tree.query(parcel_coords, k=1)
    gdf['DIST_TO_RAIL_LUAS_M'] = dist_rail
else:
    gdf['DIST_TO_RAIL_LUAS_M'] = np.nan

# Overall nearest transit distance (minimum across both modes)
gdf['DIST_TO_ANY_TRANSIT_M'] = np.minimum(gdf['DIST_TO_BUS_M'], gdf['DIST_TO_RAIL_LUAS_M'])

# Local transit density: count of bus stops within 500m walking radius (statutory RZLT benchmark)
stops_within_500m = bus_tree.query_ball_point(parcel_coords, r=500.0)
gdf['BUS_STOPS_WITHIN_500M'] = [len(neighbors) for neighbors in stops_within_500m]

# Log-transformed distance features to stabilize heavy right-skew in rural/edge parcels
gdf['LOG_DIST_BUS'] = np.log1p(gdf['DIST_TO_BUS_M'])
gdf['LOG_DIST_RAIL_LUAS'] = np.log1p(gdf['DIST_TO_RAIL_LUAS_M'])

In [7]:
# ==============================================================================
# 4. SUMMARY AUDIT & SAVE ENRICHED DATASET
# ==============================================================================
print("\n=== Multi-Modal Transit Distance Summary (Meters) ===")
summary_cols = ['DIST_TO_BUS_M', 'DIST_TO_RAIL_LUAS_M', 'DIST_TO_ANY_TRANSIT_M', 'BUS_STOPS_WITHIN_500M']
print(gdf[summary_cols].describe().round(1))

print("\n=== Median Transit Metrics by Target Status (1 = Included, 0 = Excluded) ===")
print(gdf.groupby('DECISION_STATUS')[summary_cols].median().round(1))

output_path = os.path.join("..", "Datasets", "RZLT_Samples", "RZLT_AllCodes_10to1_WithGTFS.geojson")
gdf.to_file(output_path, driver="GeoJSON")
print(f"\nEnriched dataset saved -> {output_path}")


=== Multi-Modal Transit Distance Summary (Meters) ===
       DIST_TO_BUS_M  DIST_TO_RAIL_LUAS_M  DIST_TO_ANY_TRANSIT_M  \
count         6303.0               6303.0                 6303.0   
mean           267.6               6590.4                  262.9   
std            276.5              11103.2                  254.8   
min              5.6                 82.3                    5.6   
25%            114.1               1441.5                  114.0   
50%            194.4               2531.3                  194.2   
75%            326.1               5239.1                  322.6   
max           4742.5             105920.3                 4566.7   

       BUS_STOPS_WITHIN_500M  
count                 6303.0  
mean                     7.0  
std                      5.5  
min                      0.0  
25%                      2.0  
50%                      6.0  
75%                     10.0  
max                     77.0  

=== Median Transit Metrics by Target Status (1 = Inc